# Pakistani Car Damage AI — train_colab.ipynb (T4 GPU training)

**Goal:** train a REAL fine-tuned YOLOv8n-seg car-part segmentation model on Carparts-Seg and automatically save `best.pt` + metrics + samples to Google Drive.

**Model decision (verified, see validation cell): YOLOv8n-seg — NOT plain YOLOv8n detection.**
Each label line is `class x1 y1 x2 y2 ...` (a polygon, 3+ points). A detection model uses only boxes and would throw the polygon shape away. A `-seg` model learns the actual part shapes. Evidence: full local check found 20,374 polygon instances, token counts 7/9/11/13… per line, all coordinates in [0, 1], 23 classes, 0 images without a label file.

**Honesty box:**
- `yolov8n.pt` in the project (COCO car detector) is **NOT** this project's trained model. The project model is the `best.pt` produced by THIS notebook.
- Until this notebook finishes, all part-segmentation metrics are **TBD**. Do not invent them.
- Evaluation in this notebook is labelled **PRELIMINARY**: 191 cross-split pHash candidate pairs are still unconfirmed (28 at distance ≤6, 163 at 7–12; human-review draft file missing), so the shipped test split is NOT claimed leakage-free.
- Licence: original Roboflow project states CC BY 4.0, but the curated 3,833-image set states no dataset licence — commercial rights are **NOT confirmed**. Academic use with attribution only.

**Before you run:** Colab → Runtime → Change runtime type → **GPU (T4)**. This notebook checks the GPU and stops with a warning if no GPU is found. Expected time on T4 (~15 GB VRAM): 2-epoch smoke test ~5–15 min; full 25 epochs ~1–2 hours (15 epochs if time is short, 30 if time allows — edit `FULL_EPOCHS` below).

In [ ]:
# Cell 1 — GPU check (T4, ~15 GB VRAM expected) + install ultralytics
import torch, sys
print('Python:', sys.version)
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM total (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
else:
    print('WARNING: No GPU. Runtime -> Change runtime type -> GPU (T4), then re-run.')
!nvidia-smi -L 2>/dev/null || echo 'nvidia-smi not available'
%pip install -q ultralytics
from ultralytics import YOLO
print('ultralytics imported OK')

In [ ]:
# Cell 2 — Dataset source: Google Drive ZIP first, official Ultralytics download as fallback.
# Drive route (recommended by the task): upload carparts-seg.zip to Drive first, e.g. MyDrive/CarDamageAI/carparts-seg.zip.
# Fallback route (no Drive upload needed): if the Drive zip is missing, Colab downloads the SAME official zip
# from Ultralytics (https://github.com/ultralytics/assets/releases/download/v0.0.0/carparts-seg.zip, ~133-134 MB).
from google.colab import drive
drive.mount('/content/drive')

USE_DRIVE_ZIP = True   # <-- set False to skip Drive and download directly from Ultralytics
DRIVE_ZIP = '/content/drive/MyDrive/CarDamageAI/carparts-seg.zip'
DRIVE_EXPORT = '/content/drive/MyDrive/CarDamageAI/exports'
OFFICIAL_URL = 'https://github.com/ultralytics/assets/releases/download/v0.0.0/carparts-seg.zip'

import os, subprocess
os.makedirs(DRIVE_EXPORT, exist_ok=True)
LOCAL_ZIP = '/content/carparts-seg.zip'
if USE_DRIVE_ZIP and os.path.exists(DRIVE_ZIP):
    import shutil; shutil.copy(DRIVE_ZIP, LOCAL_ZIP)
    print('Using Drive zip:', DRIVE_ZIP)
elif os.path.exists(LOCAL_ZIP):
    print('Reusing already-downloaded zip:', LOCAL_ZIP)
else:
    print('Drive zip not found — downloading from official Ultralytics URL...')
    subprocess.run(['curl', '-L', '-o', LOCAL_ZIP, OFFICIAL_URL], check=True)
    print('Downloaded:', LOCAL_ZIP)
print('Zip size (bytes):', os.path.getsize(LOCAL_ZIP))
assert os.path.getsize(LOCAL_ZIP) > 100_000_000, 'Zip looks too small (<100 MB) — download may have failed. Check the file and re-run.'
print('Export folder:', DRIVE_EXPORT)

In [ ]:
# Cell 3 — Unzip to /content (a COPY — Drive zip and original project dataset are never modified)
import zipfile, pathlib
WORK = pathlib.Path('/content/carparts')
if not (WORK / 'images').exists():
    print('Unzipping...')
    with zipfile.ZipFile('/content/carparts-seg.zip') as z:
        z.extractall('/content/carparts')
else:
    print('Already unzipped, reusing /content/carparts')
for split in ('train', 'val', 'test'):
    print(split, 'images:', len(list((WORK / 'images' / split).glob('*.jpg'))), 'labels:', len(list((WORK / 'labels' / split).glob('*.txt'))))

In [ ]:
# Cell 4 — VALIDATION (do not skip). Segmentation format, matching, empty labels, thin classes.
# Correction to an earlier check: 4-point polygons (9 tokens) are VALID segmentation polygons, not errors.
# A polygon needs at least 3 points (7 tokens). Only <7 tokens, odd coordinate counts, or coords outside [0,1] are bad.
import pathlib, collections, json, shutil
WORK = pathlib.Path('/content/carparts'); QUAR = pathlib.Path('/content/carparts_quarantine'); QUAR.mkdir(exist_ok=True)
CLASS_NAMES = ['back_bumper','back_door','back_glass','back_left_door','back_left_light','back_light','back_right_door','back_right_light','front_bumper','front_door','front_glass','front_left_door','front_left_light','front_light','front_right_door','front_right_light','hood','left_mirror','object','right_mirror','tailgate','trunk','wheel']
report = {}
for split in ('train', 'val', 'test'):
    img_dir, lbl_dir = WORK / 'images' / split, WORK / 'labels' / split
    imgs = {p.stem for p in img_dir.glob('*.jpg')}; lbls = {p.stem for p in lbl_dir.glob('*.txt')}
    counts, empty, bad, small4 = collections.Counter(), [], [], 0
    for lbl in sorted(lbl_dir.glob('*.txt')):
        lines = [l for l in lbl.read_text().strip().splitlines() if l.strip()]
        if not lines: empty.append(lbl.name); continue
        for line in lines:
            t = line.split(); coords = [float(x) for x in t[1:]]
            if len(t) == 9: small4 += 1
            if len(t) < 7 or len(coords) % 2 != 0 or min(coords) < 0 or max(coords) > 1:
                bad.append((lbl.name, line[:80]))
            else: counts[int(float(t[0]))] += 1
    report[split] = {'images': len(imgs), 'labels': len(lbls), 'matched_stems': len(imgs & lbls),
                     'images_without_label': len(imgs - lbls), 'labels_without_image': len(lbls - imgs),
                     'empty_label_files': len(empty), 'bad_lines': len(bad), 'four_point_polygons': small4,
                     'instances_by_class_id': dict(sorted(counts.items()))}
    print(split, report[split])
    qdir = QUAR / split; qdir.mkdir(exist_ok=True)
    for name in empty:
        stem = name[:-4]
        for src in (lbl_dir / name, img_dir / f'{stem}.jpg'):
            if src.exists(): shutil.move(str(src), str(qdir / src.name))
    print(f'  -> quarantined {len(empty)} empty-label pairs from {split} in the /content copy only.')
print()
print('EXPECTED: train 3156 / val 401 / test 276; matched = images in every split; empty total 135 (116/12/7); bad lines 0.')
print('Thin classes (full dataset): tailgate ~101 instances, object ~12 — per-class metrics for these will be unreliable.')
pathlib.Path('/content/validation_report.json').write_text(json.dumps(report, indent=2))
shutil.copy('/content/validation_report.json', f'{DRIVE_EXPORT}/validation_report.json')
print('Validation report saved to Drive exports.')

In [ ]:
# Cell 5 — Leakage documentation (read-only record — NO split correction is done in this notebook)
import pathlib
note = '''# Cross-split leakage candidates — documented, NOT corrected in this notebook
- Source: pilot-results/leakage-review/ (191 cross-split pHash candidate pairs).
- Band ≤6 (stronger candidates): 28 pairs (train-val 15, train-test 13, val-test 0).
- Band 7-12 (weaker candidates): 163 pairs (train-val 90, train-test 61, val-test 12).
- Human review status: 13 pairs marked Uncertain by the reviewer; the full per-pair draft decisions file
  (LEAKAGE_HUMAN_REVIEW_DECISIONS_DRAFT.csv) was NOT received, so no pair is confirmed or cleared here.
- Consequence: the shipped train/val/test splits are used UNCHANGED in this notebook (originals untouched,
  no deletion/exclusion/regrouping beyond the empty-label quarantine in the /content copy).
- Therefore ALL evaluation metrics from this notebook are labelled PRELIMINARY and must be reported as
  "on the shipped test split, with candidate leakage unresolved" — never as leakage-free results.
- Future step (separate approval required): human re-review of the 13 uncertain pairs + draft file, then a
  separately approved split rebuild if confirmed leakage requires it.
'''
pathlib.Path('/content/LEAKAGE_NOTE.md').write_text(note)
import shutil; shutil.copy('/content/LEAKAGE_NOTE.md', f'{DRIVE_EXPORT}/LEAKAGE_NOTE.md')
print(note)

In [ ]:
# Cell 6 — Write the Colab training config (correct paths + 23 class names; do NOT use the zip's shipped yaml as-is — its split comments are wrong)
import pathlib
names_yaml = '\n'.join(f'  {i}: {n}' for i, n in enumerate(['back_bumper','back_door','back_glass','back_left_door','back_left_light','back_light','back_right_door','back_right_light','front_bumper','front_door','front_glass','front_left_door','front_left_light','front_light','front_right_door','front_right_light','hood','left_mirror','object','right_mirror','tailgate','trunk','wheel']))
yaml_text = f'# Generated by train_colab.ipynb from the validated /content copy.\npath: /content/carparts\ntrain: images/train\nval: images/val\ntest: images/test\nnames:\n{names_yaml}\n'
pathlib.Path('/content/colab_data.yaml').write_text(yaml_text)
print(yaml_text)

## Training — smoke test first, then full run

- `RUN_SMOKE = True` runs **2 epochs** (name `smoke`) to prove the pipeline works end to end. If it passes, you know data, GPU, and saving all work.
- `FULL_EPOCHS` is yours to choose: **15** if time is short, **25** (default) as a balanced choice, **30** if time allows. The full run starts fresh from `yolov8n-seg.pt` (not from the smoke checkpoint) so its metrics stay clean.
- Batch 16 on T4 (~15 GB VRAM). If you get an out-of-memory error, set `BATCH = 8` and re-run that cell only.
- Do not close the Colab tab while training runs. Checkpoints are saved under `/content/runs/`.

In [ ]:
# Cell 7 — SMOKE TEST: 2 epochs (pipeline proof, not the final model)
from ultralytics import YOLO
RUN_SMOKE = True
BATCH = 16  # <-- set to 8 if you run out of GPU memory
if RUN_SMOKE:
    smoke = YOLO('yolov8n-seg.pt')
    smoke.train(data='/content/colab_data.yaml', epochs=2, imgsz=640, batch=BATCH,
                device=0, project='/content/runs', name='smoke', patience=10, save=True, plots=True)
    print('SMOKE TEST PASSED. Smoke weights: /content/runs/smoke/weights/best.pt (do NOT submit this as the final model).')
else:
    print('Smoke test skipped by user (RUN_SMOKE=False).')

In [ ]:
# Cell 8 — FULL TRAINING: 15-30 epochs (this run produces the submission model)
from ultralytics import YOLO
RUN_FULL = True
FULL_EPOCHS = 25  # <-- choose 15, 25, or 30 depending on remaining time before the 31 Oct night deadline
BATCH = 16  # <-- set to 8 if you run out of GPU memory
if RUN_FULL:
    model = YOLO('yolov8n-seg.pt')  # fresh start from pretrained seg weights (not from the smoke run)
    model.train(data='/content/colab_data.yaml', epochs=FULL_EPOCHS, imgsz=640, batch=BATCH,
                device=0, project='/content/runs', name='carparts-seg', patience=10, save=True, plots=True)
    print(f'FULL TRAINING FINISHED ({FULL_EPOCHS} epochs). Best weights: /content/runs/carparts-seg/weights/best.pt')
else:
    print('Full training skipped by user (RUN_FULL=False). Evaluation below will use the smoke model and must be labelled as such.')

In [ ]:
# Cell 9 — EVALUATION (PRELIMINARY — see leakage note). Val + held-out test, metrics SAVED to Drive.
import json, pathlib
from ultralytics import YOLO
import os
full_best = pathlib.Path('/content/runs/carparts-seg/weights/best.pt')
smoke_best = pathlib.Path('/content/runs/smoke/weights/best.pt')
BEST = full_best if full_best.exists() else smoke_best
RUN_TYPE = 'FULL' if full_best.exists() else 'SMOKE-ONLY (not the final model)'
assert BEST.exists(), 'No best.pt found — neither full nor smoke training finished. Do not proceed.'
print('Evaluating:', BEST, '| run type:', RUN_TYPE)
model = YOLO(str(BEST))
metrics = {'evaluation_status': 'PRELIMINARY - shipped test split, cross-split leakage candidates unresolved (see LEAKAGE_NOTE.md)',
           'evaluated_weights': str(BEST), 'run_type': RUN_TYPE}
for split in ('val', 'test'):
    m = model.val(data='/content/colab_data.yaml', split=split, imgsz=640, verbose=False)
    d = {'precision': round(float(m.box.mp), 4), 'recall': round(float(m.box.mr), 4),
         'mAP50_box': round(float(m.box.map50), 4), 'mAP50-95_box': round(float(m.box.map), 4),
         'mAP50_mask': round(float(m.seg.map50), 4) if hasattr(m, 'seg') and m.seg is not None else None,
         'per_class_mAP50_box': [round(float(v), 4) for v in m.box.maps]}
    metrics[split] = d; print(split, d)
pathlib.Path('/content/evaluation_metrics.json').write_text(json.dumps(metrics, indent=2))
import shutil; shutil.copy('/content/evaluation_metrics.json', f'{DRIVE_EXPORT}/evaluation_metrics.json')
print('Metrics saved to Drive exports. Copy these numbers EXACTLY into the report — do not round up or invent others.')

In [ ]:
# Cell 10 — SAMPLE PREDICTIONS (5 held-out test images, annotated + JSON, saved to Drive)
import pathlib, json
from ultralytics import YOLO
from IPython.display import Image as IPyImage, display
model = YOLO(str(BEST))
out = pathlib.Path('/content/sample_outputs'); out.mkdir(exist_ok=True)
samples = []
for img in sorted(pathlib.Path('/content/carparts/images/test').glob('*.jpg'))[:5]:
    r = model.predict(str(img), imgsz=640, conf=0.25, save=False, verbose=False)[0]
    dets = [(model.names[int(c)], round(float(p), 3)) for c, p in zip(r.boxes.cls.tolist(), r.boxes.conf.tolist())] if r.boxes is not None and len(r.boxes) else []
    samples.append({'image': img.name, 'detections': dets})
    print(img.name, '->', dets if dets else 'no parts detected')
    annotated = out / f'annotated_{img.name}'
    r.save(filename=str(annotated)); display(IPyImage(filename=str(annotated)))
pathlib.Path('/content/sample_predictions.json').write_text(json.dumps(samples, indent=2))
print('Samples saved to /content/sample_outputs/ and sample_predictions.json')

In [ ]:
# Cell 11 — AUTOMATIC SAVE TO DRIVE (weights, results, metrics, samples, summary). Run this cell last.
import pathlib, shutil, os, json
RUN_DIR = pathlib.Path('/content/runs/carparts-seg') if pathlib.Path('/content/runs/carparts-seg/weights/best.pt').exists() else pathlib.Path('/content/runs/smoke')
src = RUN_DIR / 'weights' / 'best.pt'
assert src.exists(), 'best.pt NOT FOUND — training did not finish. Do not proceed to the report.'
dst = pathlib.Path(DRIVE_EXPORT) / 'best.pt'; shutil.copy(src, dst)
for f in ('results.csv', 'results.png'):
    p = RUN_DIR / f
    if p.exists(): shutil.copy(p, pathlib.Path(DRIVE_EXPORT) / f)
for f in ('evaluation_metrics.json', 'validation_report.json', 'LEAKAGE_NOTE.md', 'sample_predictions.json'):
    p = pathlib.Path('/content') / f
    if p.exists(): shutil.copy(p, pathlib.Path(DRIVE_EXPORT) / f)
shutil.copytree('/content/sample_outputs', pathlib.Path(DRIVE_EXPORT) / 'sample_outputs', dirs_exist_ok=True)
# Training summary (generated from measured files only)
metrics = json.loads(pathlib.Path('/content/evaluation_metrics.json').read_text())
summary = f'''# Training summary (auto-generated by train_colab.ipynb)
- Run type: {metrics.get('run_type')} | Weights: {src} ({os.path.getsize(src)} bytes) -> Drive: {dst} ({os.path.getsize(dst)} bytes)
- Evaluation status: {metrics.get('evaluation_status')}
- Val metrics: {metrics.get('val')}
- Test metrics: {metrics.get('test')}
- Next step: download best.pt from Drive, place it at models/part_detection/best.pt in the project, then run predict.py (see Cell 12).
'''
pathlib.Path(DRIVE_EXPORT, 'TRAINING_SUMMARY.md').write_text(summary)
print(summary)
print('Drive export folder contents:')
for p in sorted(pathlib.Path(DRIVE_EXPORT).rglob('*')):
    if p.is_file(): print(' ', p.relative_to(DRIVE_EXPORT), p.stat().st_size, 'bytes')
# Final verification: exported best.pt loads as a segmentation model with 23 classes
m = YOLO(str(dst)); print('Verified exported model — task:', m.task, '| classes:', len(m.names))

## After training — integrate best.pt into the existing inference workflow (do this on Windows / VS Code)

1. Download `best.pt` from your Drive export folder (`MyDrive/CarDamageAI/exports/best.pt`).
2. In the project, copy it to **exactly** `models/part_detection/best.pt` (same folder, same name). Do not overwrite `yolov8n.pt`, do not rename it.
3. Run: `python predict.py --image uploads/images/honda_civic_cc0.jpg --output-dir outputs/predict_test`
   Check `prediction.json`: `part_model_status` must say `"TRAINED"` and part detections must list real part names + confidences. If it still says `NOT_TRAINED`, the file is in the wrong folder.
4. Run the API locally: `uvicorn api.main:app --host 0.0.0.0 --port 8000`, open `http://localhost:8000/health` — it must say `part_model_status: TRAINED`.
5. Copy numbers from `evaluation_metrics.json` (Drive exports) into `docs/submission/REPORT_DRAFT.md` and the presentation outline, replacing every TBD. Keep the word PRELIMINARY next to test metrics until leakage is resolved.
6. Upload `best.pt` as a GitHub Release asset yourself, paste the link into `SUBMISSION_README.md` and `frontend/config.js`, then deploy the API and Vercel frontend yourself (see `docs/submission/DEPLOYMENT_GUIDE.md`).
7. Full beginner steps (Windows + Colab + after-training) are in `docs/submission/VS_CODE_AFTER_TRAINING.md` and `docs/submission/RUN_INSTRUCTIONS_WINDOWS_COLAB.md`.